# Tema 9. Pandas

Cada celda de código corresponde al script del mismo nombre. La primera celda cambia el directorio de trabajo a la raíz del repositorio.

In [ ]:
import os

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")
print(os.getcwd())

## Series y DataFrame

Script `series_dataframe.py`.

In [ ]:
import pandas as pd

# Una Series es una columna con etiquetas
precios = pd.Series([2450.0, 1890.0, 1320.9], name="precio")
print(precios)
print(precios.dtype)
print(precios.mean())

# Un DataFrame reúne varias columnas con nombre y tipos distintos
equipos = pd.DataFrame({
    "codigo": ["A001", "A003", "A005"],
    "equipo": ["Servidor web", "Switch planta 1", "Portátil dirección"],
    "precio": [2450.0, 1890.0, 1320.9],
})
print(equipos)
print(equipos.shape)
print(list(equipos.columns))
print(equipos.dtypes)

## Explorar un DataFrame

Script `explorar_activos.py`.

In [ ]:
import pandas as pd

activos = pd.read_csv("Tema09/datos/activos.csv")
# Pandas oculta columnas intermedias (...) cuando no caben en la pantalla
print(activos.head())
print(activos[["codigo", "equipo", "precio", "estado"]].head())
print(activos.shape)
activos.info()
print(activos["precio"].describe())
print(activos["estado"].value_counts())
print(activos["fabricante"].value_counts().head(3))

## Parámetros de lectura y guardado

Script `leer_parametros.py`.

In [ ]:
import pandas as pd

# El fichero lo ha guardado Excel: separador ;, coma decimal y cp1252
inventario = pd.read_csv(
    "Tema09/datos/inventario_excel.csv",
    sep=";",
    decimal=",",
    encoding="cp1252",
)
print(inventario)
print(inventario.dtypes)

# Guardamos sin y con el índice, y leemos los dos ficheros
inventario.to_csv("Tema09/inventario_utf8.csv", index=False)
inventario.to_csv("Tema09/inventario_indice.csv")
sin_indice = pd.read_csv("Tema09/inventario_utf8.csv")
con_indice = pd.read_csv("Tema09/inventario_indice.csv")
print(sin_indice.head(3))
print(con_indice.head(3))

## Selección con loc e iloc

Script `seleccionar.py`.

In [ ]:
import pandas as pd

activos = pd.read_csv("Tema09/datos/activos.csv")

# Columnas
print(activos["precio"].head(3))
print(activos[["codigo", "precio"]].head(3))

# loc usa etiquetas e incluye el final; iloc usa posiciones y no lo incluye
print(activos.loc[2, "equipo"])
print(activos.loc[0:2, ["codigo", "equipo"]])
print(activos.iloc[0:2, 0:2])

# Con set_index() el código pasa a ser la etiqueta de cada fila
indexado = activos.set_index("codigo")
print(indexado.loc["A005", "equipo"])
print(indexado.loc["A001":"A003", "precio"])

## Filtros

Script `filtros.py`.

In [ ]:
import pandas as pd

activos = pd.read_csv("Tema09/datos/activos.csv")

# Condición simple
mascara = activos["precio"] > 4000
print(mascara.sum())
print(activos[mascara][["codigo", "equipo", "precio"]])

# Condiciones compuestas: & (y), | (o), ~ (no), cada una entre paréntesis
en_madrid = (activos["ubicacion"] == "Madrid") & (activos["estado"] == "Activo")
print(activos[en_madrid][["codigo", "equipo", "ubicacion", "estado"]])

# isin() y str.contains()
print(activos[activos["fabricante"].isin(["Dell", "HP"])][["codigo", "fabricante"]])
print(activos[activos["equipo"].str.contains("Portátil")][["codigo", "equipo"]])

## Copias y asignaciones

Script `copias.py`.

In [ ]:
import pandas as pd

activos = pd.read_csv("Tema09/datos/activos.csv")

# Un subconjunto es una copia: modificarlo no cambia el original
caros = activos[activos["precio"] > 4000]
caros["estado"] = "Revisar"
print(caros[["codigo", "estado"]])
print(activos.loc[[1, 7, 15, 16], ["codigo", "estado"]])

# Para modificar el original se asigna con loc, en un solo paso
retirados = activos["estado"] == "Retirado"
activos.loc[retirados, "estado"] = "Baja"
print(activos["estado"].value_counts())

## Ordenar y calcular

Script `ordenar_calcular.py`.

In [ ]:
import numpy as np
import pandas as pd

activos = pd.read_csv("Tema09/datos/activos.csv")

# Ordenar por una columna y por varias
mas_caros = activos.sort_values("precio", ascending=False)
print(mas_caros[["codigo", "equipo", "precio"]].head(3))
orden = activos.sort_values(["ubicacion", "precio"], ascending=[True, False])
print(orden[["codigo", "ubicacion", "precio"]].head(5))

# Columna calculada, sin bucles
activos["precio_iva"] = (activos["precio"] * 1.21).round(2)
print(activos[["codigo", "precio", "precio_iva"]].head(3))

# Columna con una condición
activos["gama"] = np.where(activos["precio"] >= 2000, "Alta", "Estándar")
print(activos[["codigo", "precio", "gama"]].head(3))
print(activos["gama"].value_counts())

## Agregaciones y groupby

Script `agregaciones.py`.

In [ ]:
import pandas as pd

activos = pd.read_csv("Tema09/datos/activos.csv")

# Resumen de una columna
precio = activos["precio"]
print(precio.sum(), precio.mean().round(2), precio.min(), precio.max())

# Una fila de resultados por cada grupo
print(activos.groupby("ubicacion")["precio"].mean().round(2))
print(activos.groupby("estado")["precio"].agg(["count", "sum"]))

# reset_index() devuelve el resultado como un DataFrame normal
por_fabricante = activos.groupby("fabricante")["precio"].sum().reset_index()
print(por_fabricante.sort_values("precio", ascending=False).head(3))

## Ausentes y duplicados

Script `ausentes_duplicados.py`.

In [ ]:
import pandas as pd

sucio = pd.read_csv("Tema09/datos/activos_sucio.csv")
print(sucio.shape)

# Ausentes: cuántos hay por columna y en qué filas
print(sucio.isna().sum())
print(sucio[sucio["ubicacion"].isna()][["codigo", "equipo", "ubicacion"]])

# fillna() y dropna() devuelven un resultado nuevo
sucio["ubicacion"] = sucio["ubicacion"].fillna("Sin asignar")
print(sucio[sucio["codigo"].isin(["A006", "A021"])][["codigo", "ubicacion"]])
print(sucio["ubicacion"].value_counts())

# Duplicados: filas repetidas por completo
print(sucio.duplicated().sum())
print(sucio[sucio.duplicated()][["codigo", "equipo", "precio"]])
sucio = sucio.drop_duplicates()
print(sucio.shape)
print(sucio[sucio["codigo"] == "A005"][["codigo", "equipo", "precio"]])

## Tipos y texto

Script `tipos_texto.py`.

In [ ]:
import pandas as pd

sucio = pd.read_csv("Tema09/datos/activos_sucio.csv")

# Texto: quitar espacios y unificar mayúsculas
print(sucio["fabricante"].value_counts().head(4))
sucio["fabricante"] = sucio["fabricante"].str.strip().str.lower()
sucio["estado"] = sucio["estado"].str.strip().str.lower()
print(sucio["fabricante"].value_counts().head(4))
print(sucio["estado"].value_counts())

# Tipos: precio es texto porque contiene 1.6k y N/D
print(sucio["precio"].dtype)
sucio["precio_num"] = pd.to_numeric(sucio["precio"], errors="coerce")
print(sucio["precio_num"].dtype)
print(sucio[sucio["precio_num"].isna()][["codigo", "precio", "precio_num"]])

## Fechas y rangos

Script `fechas.py`.

In [ ]:
import pandas as pd

sucio = pd.read_csv("Tema09/datos/activos_sucio.csv")
sucio = sucio.drop_duplicates()
print(sucio["Fecha Compra"].dtype)

# errors="coerce" convierte las fechas imposibles en NaT
sucio["fecha"] = pd.to_datetime(sucio["Fecha Compra"], errors="coerce")
print(sucio["fecha"].dtype)
print(sucio[sucio["fecha"].isna()][["codigo", "Fecha Compra", "fecha"]])

# El accesor dt da el año, y las comparaciones filtran por rango
sucio["anio"] = sucio["fecha"].dt.year
print(sucio["anio"].value_counts().sort_index())
recientes = sucio[sucio["fecha"] >= "2023-01-01"]
print(recientes[["codigo", "fecha"]].head(5))

## Informe de tickets

Script `informe_tickets.py`.

In [ ]:
import pandas as pd

# N/D se lee como valor ausente
tickets = pd.read_csv("Tema09/datos/tickets.csv", na_values=["N/D"])
print(tickets.head(8))
print(len(tickets), "tickets")
print("Sin resolver:", tickets["horas_resolucion"].isna().sum())
print("Media global:", round(tickets["horas_resolucion"].mean(), 2), "h")

# Resumen por técnico: número de tickets y horas medias
por_tecnico = tickets.groupby("tecnico")["horas_resolucion"].agg(
    ["count", "mean"]
)
por_tecnico = por_tecnico.round(1).sort_values("mean", ascending=False)
print(por_tecnico)

# Resumen por prioridad
print(tickets.groupby("prioridad")["horas_resolucion"].mean().round(1))

# Guardar el informe sin el índice numérico
por_tecnico.reset_index().to_csv("Tema09/informe_tickets.csv", index=False)

## Bucle para vectorizar

Script `iva_bucle.py`.

In [ ]:
import pandas as pd

inventario = pd.DataFrame({"precio": [2450.0, 1890.0, 1320.9]})
con_iva = []
for precio in inventario["precio"]:
    con_iva.append(round(precio * 1.21, 1))
print(con_iva)

## Ejercicio propio

Agrega debajo una celda que cargue `Tema09/datos/tickets.csv` con `na_values=["N/D"]` y muestre los tickets de prioridad Alta sin resolver, con las columnas `codigo`, `prioridad` y `tecnico`.